# VOID GEN FULL · Colab

在「執行階段 → 變更執行階段類型」選 GPU，然後依序執行。先備妥同一份交付中的 `VOID-Colab-Source.zip`。第一次啟動需要上傳原始碼包並下載模型。作品會直接寫到 `我的雲端硬碟/VOID-Colab/output`。

這是單人工作區。使用 Colab 的私人連線入口。

In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, subprocess, sys, zipfile

drive.mount('/content/drive')
HOME = Path('/content/drive/MyDrive/VOID-Colab')
HOME.mkdir(exist_ok=True)
BUNDLE = HOME / 'VOID-Colab-Source.zip'
if not BUNDLE.exists():
    uploaded = files.upload()
    if BUNDLE.name not in uploaded:
        raise ValueError(f'請上傳 {BUNDLE.name}')
    BUNDLE.write_bytes(uploaded[BUNDLE.name])
SOURCE = Path('/content/void.gen')
SOURCE.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    archive.extractall(SOURCE)
COMFY = Path('/content/ComfyUI')

def run(*args, **kwargs):
    return subprocess.run(args, check=True, **kwargs)

run('nvidia-smi')

## 安裝 ComfyUI、節點與模型

預設下載 SDXL。把需要的項目加入 `PROFILES` 再執行。模型會保存在 Drive，下一次可重用。選用的 Nova Anime、Moody Moss 和 DesireX 可依模型表手動補齊。

In [ ]:
import json, re

PROFILES = ['sdxl']  # 例如 ['sdxl', 'klein', 'qwen', 'moss', 'wan', 'florence', 'wd14']
SECTIONS = {'sdxl':'SDXL', 'klein':'FLUX.2', 'qwen':'Qwen', 'moss':'MOSS', 'wan':'Wan', 'florence':'Florence-2'}
NODES = {
    'gguf': ('leejet/ComfyUI-GGUF', ''),
    'moss': ('T8mars/Comfyui-Moss-Image-T8', 'c045ac07e48b202d8f9df76992b8446c0cea663b'),
    'florence': ('kijai/ComfyUI-Florence2', '9ece3de914214c5f581d725167bc9d0eeb0d1120'),
    'wd14': ('pythongosssss/ComfyUI-WD14-Tagger', ''),
}
if unknown := set(PROFILES) - SECTIONS.keys() - {'wd14'}:
    raise ValueError(unknown)
if not COMFY.exists():
    run('git', 'clone', 'https://github.com/Comfy-Org/ComfyUI.git', str(COMFY))
run(sys.executable, '-m', 'pip', 'install', '-r', str(COMFY / 'requirements.txt'), 'aiohttp', 'Pillow')
plugins = set(PROFILES) & NODES.keys()
if set(PROFILES) & {'klein', 'qwen', 'wan'}:
    plugins.add('gguf')
for key in sorted(plugins):
    repo, revision = NODES[key]
    target = COMFY / 'custom_nodes' / repo.split('/')[-1]
    if not target.exists():
        run('git', 'clone', 'https://github.com/' + repo + '.git', str(target))
    if revision:
        run('git', '-C', str(target), 'checkout', '--detach', revision)
    if (target / 'requirements.txt').exists():
        run(sys.executable, '-m', 'pip', 'install', '-r', str(target / 'requirements.txt'))

models = HOME / 'models'
folders = {'checkpoints','diffusion_models','text_encoders','vae','loras','LLM','upscale_models'}
for folder in folders:
    (models / folder).mkdir(parents=True, exist_ok=True)
    local = COMFY / 'models' / folder
    if local.exists() and not local.is_symlink():
        for entry in local.iterdir():
            if not (models / folder / entry.name).exists():
                shutil.copy2(entry, models / folder / entry.name)
        shutil.rmtree(local)
    if not local.is_symlink():
        local.symlink_to(models / folder, target_is_directory=True)

manual = (SOURCE / 'docs/models-v4.md').read_text(encoding='utf-8')
for section in manual.split('\n## ')[1:]:
    if not any(section.startswith(SECTIONS[key] + ' ') for key in PROFILES if key in SECTIONS):
        continue
    for name, url, folder in re.findall(r'^\| \[([^\]]+)\]\((https://huggingface\.co/[^)]+)\) \| `([\w/-]+)/` \|$', section, re.M):
        destination = models / folder / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.is_file() and destination.stat().st_size:
            continue
        partial = destination.with_name(destination.name + '.part')
        run('curl', '--fail', '--location', '--retry', '3', '--continue-at', '-', '--output', str(partial), url)
        partial.replace(destination)
print('模型準備完成')

## 啟動 VOID

這格會先等 ComfyUI 就緒，再提供桌面版和手機版網址。建議先用 SDXL 生成一張，確認 Drive 出現成品。

In [ ]:
import subprocess, time, urllib.request
from google.colab.output import eval_js
from IPython.display import HTML, display
from html import escape

RUNTIME = Path('/content/void-runtime')
RUNTIME.mkdir(exist_ok=True)
OUTPUT = HOME / 'output'
OUTPUT.mkdir(exist_ok=True)
assert not any(globals().get(key) and globals()[key].poll() is None for key in ('comfy_process', 'void_process')), '服務已啟動'
def start(name, cwd, *args):
    with (RUNTIME / (name + '.log')).open('wb') as log:
        return subprocess.Popen([sys.executable, '-u', *args], cwd=cwd, stdout=log, stderr=subprocess.STDOUT)
def wait_http(process, url, logfile):
    for _ in range(120):
        if process.poll() is not None:
            break
        try:
            with urllib.request.urlopen(url, timeout=2) as response:
                if response.status == 200:
                    return
        except OSError:
            time.sleep(1)
    print((RUNTIME / logfile).read_text(errors='replace')[-7000:])
    raise RuntimeError(f'服務啟動失敗：{url}')

comfy_process = start('comfy', COMFY, 'main.py', '--listen', '127.0.0.1', '--port', '8188',
                      '--output-directory', str(OUTPUT))
try:
    wait_http(comfy_process, 'http://127.0.0.1:8188/system_stats', 'comfy.log')
    void_process = start('void', SOURCE, 'void_colab.py', '--drive', str(HOME),
                         '--runtime', str(RUNTIME), '--comfy', str(COMFY))
    wait_http(void_process, 'http://127.0.0.1:7860/api/status', 'void.log')
except Exception:
    comfy_process.terminate()
    if globals().get('void_process') and void_process.poll() is None:
        void_process.terminate()
    raise
url = eval_js('google.colab.kernel.proxyPort(7860)').rstrip('/')
display(HTML(f'<a href="{escape(url)}/" target="_blank" rel="noopener">開啟 VOID 桌面版</a> · <a href="{escape(url)}/mobile/" target="_blank" rel="noopener">開啟手機介面</a>'))

## 結束使用

完成任務後執行下方這格，先把圖庫資料庫備份到 Drive，再停止服務。

In [ ]:
import signal
if void_process.poll() is None:
    void_process.send_signal(signal.SIGINT)
    void_process.wait(timeout=45)
if comfy_process.poll() is None:
    comfy_process.terminate()
    comfy_process.wait(timeout=30)
print('Drive 作品目錄：', OUTPUT)